In [ ]:
!pip install requests pydantic reportlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 16.8 MB/s eta 0:00:00


In [ ]:
import sys
import os
import json
import sqlite3
import time
import textwrap
import re
from datetime import datetime
from typing import Optional, List

try:
    import requests
except ImportError:
    sys.exit("❌  'requests' not found. Run:  pip install requests")

try:
    from pydantic import BaseModel, Field
except ImportError:
    sys.exit("❌  'pydantic' not found. Run:  pip install pydantic")

try:
    from reportlab.lib.pagesizes import letter
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.units import inch
    from reportlab.lib import colors
    from reportlab.platypus import (
        SimpleDocTemplate, Paragraph, Spacer, PageBreak,
        HRFlowable, KeepTogether
    )
    from reportlab.lib.enums import TA_CENTER, TA_JUSTIFY, TA_LEFT
except ImportError:
    sys.exit("❌  'reportlab' not found. Run:  pip install reportlab")


# ─────────────────────────────────────────────
# 1. LOGGING UTILITY
# ─────────────────────────────────────────────
LOG_WIDTH = 72

def log(msg: str, level: str = "INFO"):
    ts = datetime.now().strftime("%H:%M:%S")
    prefix = {"INFO": "ℹ", "OK": "✅", "WARN": "⚠️", "ERR": "❌",
              "AGENT": "🤖", "CHAPTER": "📖", "DB": "💾",
              "PDF": "📄", "START": "🚀"}.get(level, "•")
    print(f"[{ts}] {prefix}  {msg}")

def log_section(title: str):
    print("\n" + "═" * LOG_WIDTH)
    print(f"  {title}")
    print("═" * LOG_WIDTH)

def log_banner(title: str):
    print("\n" + "█" * LOG_WIDTH)
    centered = title.center(LOG_WIDTH)
    print(centered)
    print("█" * LOG_WIDTH + "\n")


# ─────────────────────────────────────────────
# 2. PYDANTIC MODELS
# ─────────────────────────────────────────────
class Character(BaseModel):
    name: str
    role: str                     # protagonist / antagonist / supporting
    age: Optional[str] = None
    personality: str
    background: str
    goals: str
    arc: str                      # how they change over the story

class WorldInfo(BaseModel):
    setting: str
    time_period: str
    geography: str
    society: str
    rules: str                    # magic systems, tech level, laws of the world
    atmosphere: str

class PlotThread(BaseModel):
    thread_id: str
    description: str
    status: str = "active"        # active / resolved / dormant

class ChapterOutline(BaseModel):
    chapter_number: int
    title: str
    summary: str
    key_events: List[str]
    characters_present: List[str]
    plot_threads_advanced: List[str]
    emotional_tone: str

class StoryBible(BaseModel):
    title: str
    genre: str
    logline: str
    themes: List[str]
    world: WorldInfo
    characters: List[Character]
    plot_threads: List[PlotThread]
    chapter_outlines: List[ChapterOutline]

class ChapterContent(BaseModel):
    chapter_number: int
    title: str
    content: str
    word_count: int
    summary: str                  # 2-3 sentence summary for next chapter context
    plot_threads_updated: List[str]

class EditedChapter(BaseModel):
    chapter_number: int
    title: str
    content: str
    word_count: int
    editor_notes: str

class ReviewReport(BaseModel):
    overall_score: int            # 1-10
    consistency_notes: str
    pacing_notes: str
    character_arc_notes: str
    final_recommendation: str


# ─────────────────────────────────────────────
# 3. DATABASE (SQLite)
# ─────────────────────────────────────────────
DB_PATH = "novel_progress.db"

def init_db():
    log("Initialising SQLite database …", "DB")
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.execute("""
        CREATE TABLE IF NOT EXISTS story_bible (
            id INTEGER PRIMARY KEY,
            data TEXT NOT NULL,
            created_at TEXT DEFAULT CURRENT_TIMESTAMP
        )""")
    c.execute("""
        CREATE TABLE IF NOT EXISTS chapters (
            chapter_number INTEGER PRIMARY KEY,
            title TEXT,
            raw_content TEXT,
            edited_content TEXT,
            word_count INTEGER,
            summary TEXT,
            plot_threads TEXT,
            editor_notes TEXT,
            status TEXT DEFAULT 'pending',
            created_at TEXT DEFAULT CURRENT_TIMESTAMP
        )""")
    c.execute("""
        CREATE TABLE IF NOT EXISTS run_log (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            event TEXT,
            detail TEXT,
            ts TEXT DEFAULT CURRENT_TIMESTAMP
        )""")
    conn.commit()
    conn.close()
    log("Database ready.", "DB")

def save_story_bible(bible: StoryBible):
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.execute("DELETE FROM story_bible")
    c.execute("INSERT INTO story_bible (data) VALUES (?)",
              (bible.model_dump_json(),))
    conn.commit(); conn.close()
    log("Story Bible saved to DB.", "DB")

def load_story_bible() -> Optional[StoryBible]:
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.execute("SELECT data FROM story_bible ORDER BY id DESC LIMIT 1")
    row = c.fetchone(); conn.close()
    if row:
        return StoryBible.model_validate_json(row[0])
    return None

def save_chapter(ch: EditedChapter, summary: str, plot_threads: List[str]):
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.execute("""
        INSERT OR REPLACE INTO chapters
          (chapter_number, title, edited_content, word_count, summary,
           plot_threads, editor_notes, status)
        VALUES (?,?,?,?,?,?,?,'done')""",
        (ch.chapter_number, ch.title, ch.content, ch.word_count,
         summary, json.dumps(plot_threads), ch.editor_notes))
    conn.commit(); conn.close()
    log(f"Chapter {ch.chapter_number} saved to DB ({ch.word_count} words).", "DB")

def load_all_chapters() -> List[dict]:
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.execute("""SELECT chapter_number, title, edited_content, word_count,
                        summary, plot_threads, editor_notes
                 FROM chapters WHERE status='done'
                 ORDER BY chapter_number""")
    rows = c.fetchall(); conn.close()
    return [{"chapter_number": r[0], "title": r[1], "content": r[2],
             "word_count": r[3], "summary": r[4],
             "plot_threads": json.loads(r[5]), "editor_notes": r[6]}
            for r in rows]

def chapter_exists(n: int) -> bool:
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.execute("SELECT 1 FROM chapters WHERE chapter_number=? AND status='done'", (n,))
    exists = c.fetchone() is not None; conn.close()
    return exists

def db_log(event: str, detail: str = ""):
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.execute("INSERT INTO run_log (event, detail) VALUES (?,?)", (event, detail))
    conn.commit(); conn.close()


# ─────────────────────────────────────────────
# 4. OPENROUTER API WRAPPER
# ─────────────────────────────────────────────
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
AGENT_MODELS = {
    "plot": "qwen/qwen3-235b-a22b",
    "world": "qwen/qwen3-235b-a22b",

    "character": "google/gemma-3-27b-it",

    "outline": "qwen/qwen3-235b-a22b",

    "chapter":  "google/gemma-3-27b-it",

    "editor": "mistralai/mistral-small-3.2-24b-instruct",

    "reviewer": "deepseek/deepseek-r1-0528",
}
MAX_RETRIES = 3
RETRY_DELAY = 5   # seconds

def call_llm(
    api_key,
    system_prompt,
    user_prompt,
    model,
    max_tokens=10000,
    temperature=0.8
):
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
        "HTTP-Referer": "https://novel-generator.local",
        "X-Title": "Multi-Agent Novel Generator"
    }
    payload = {
    "model": model,
    "max_tokens": max_tokens,
    "temperature": temperature,
    "messages": [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]
}
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            log(f"  → API call (attempt {attempt}/{MAX_RETRIES}, max_tokens={max_tokens}) …")
            resp = requests.post(OPENROUTER_URL, headers=headers,
                                 json=payload, timeout=120)
            resp.raise_for_status()
            data = resp.json()
            text = data["choices"][0]["message"]["content"]
            tokens_used = data.get("usage", {}).get("total_tokens", "?")
            log(f"  ← Response received ({tokens_used} tokens used).")
            return text
        except requests.exceptions.Timeout:
            log(f"  Timeout on attempt {attempt}.", "WARN")
        except requests.exceptions.HTTPError as e:
            log(f"  HTTP {e.response.status_code}: {e.response.text[:200]}", "WARN")
        except Exception as e:
            log(f"  Unexpected error: {e}", "WARN")
        if attempt < MAX_RETRIES:
            log(f"  Retrying in {RETRY_DELAY}s …", "WARN")
            time.sleep(RETRY_DELAY)
    raise RuntimeError("❌  All API retries exhausted.")

def extract_json(text: str) -> dict:
    """Strip markdown fences and parse JSON."""
    text = re.sub(r"^```(?:json)?\s*", "", text.strip(), flags=re.MULTILINE)
    text = re.sub(r"\s*```$", "", text.strip(), flags=re.MULTILINE)
    text = text.strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError as e:
        # Try to find the first {...} block
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if match:
            return json.loads(match.group())
        raise ValueError(f"Could not parse JSON: {e}\nRaw text:\n{text[:500]}")


# ─────────────────────────────────────────────
# 5. AGENTS
# ─────────────────────────────────────────────

# ── 5A. PLOT AGENT ──────────────────────────
def plot_agent(api_key: str, title: str, genre: str,
               plot_seed: str, num_chapters: int) -> dict:
    log_section("PLOT AGENT — Building story structure")
    log("Generating logline, themes, and plot threads …", "AGENT")

    system = """You are a master story architect. Your job is to design compelling,
coherent plot structures for novels. Always respond with valid JSON only — no markdown,
no commentary, just the raw JSON object."""

    user = f"""Design the complete plot structure for a {genre} novel titled "{title}".
Author's seed idea: "{plot_seed or 'No seed — use your creativity.'}"
Number of chapters: {num_chapters}

Return a JSON object with exactly these keys:
{{
  "logline": "One compelling sentence describing the whole novel",
  "themes": ["theme1", "theme2", "theme3"],
  "three_act_structure": {{
    "act1_chapters": "1-{num_chapters//4}",
    "act2_chapters": "{num_chapters//4+1}-{(num_chapters*3)//4}",
    "act3_chapters": "{(num_chapters*3)//4+1}-{num_chapters}"
  }},
  "plot_threads": [
    {{"thread_id": "PT1", "description": "Main plot thread", "status": "active"}},
    {{"thread_id": "PT2", "description": "Romantic subplot", "status": "active"}},
    {{"thread_id": "PT3", "description": "Mystery subplot", "status": "active"}}
  ],
  "major_turning_points": [
    {{"chapter": 1, "event": "Opening hook"}},
    {{"chapter": {num_chapters//4}, "event": "End of Act 1 — inciting incident climax"}},
    {{"chapter": {num_chapters//2}, "event": "Midpoint reversal"}},
    {{"chapter": {(num_chapters*3)//4}, "event": "Dark night of the soul"}},
    {{"chapter": {num_chapters}, "event": "Resolution"}}
  ]
}}"""

    raw = call_llm(
    api_key,
    system,
    user,
    model=AGENT_MODELS["plot"],
    max_tokens=2000
)
    result = extract_json(raw)
    log("Plot structure built successfully.", "OK")
    db_log("plot_agent_complete", json.dumps(result)[:200])
    return result


# ── 5B. WORLD AGENT ─────────────────────────
def world_agent(api_key: str, title: str, genre: str, plot_data: dict) -> WorldInfo:
    log_section("WORLD AGENT — Building the story world")
    log("Constructing world: geography, society, atmosphere …", "AGENT")

    system = """You are a world-building expert for fiction. You craft vivid, internally
consistent fictional worlds. Always respond with valid JSON only."""

    user = f"""Build the world for a {genre} novel titled "{title}".

Story logline: {plot_data.get('logline', '')}
Themes: {', '.join(plot_data.get('themes', []))}

Return a JSON object with exactly these keys. Ensure each field's value is a single string:
{{
  "setting": "Primary setting description (2-3 sentences)",
  "time_period": "When the story takes place",
  "geography": "A single string describing the physical landscape, key locations, and scale of the world.",
  "society": "A single string describing the social structures, factions, political situation, and culture.",
  "rules": "A single string describing special rules of this world (magic, technology, laws, taboos).",
  "atmosphere": "Overall mood and tone — what does this world FEEL like"
}}"""

    raw = call_llm(api_key, system, user,model=AGENT_MODELS["world"], max_tokens=4000)
    data = extract_json(raw)
    world = WorldInfo(**data)
    log("World built successfully.", "OK")
    db_log("world_agent_complete", world.setting[:100])
    return world


# ── 5C. CHARACTER AGENT ──────────────────────
def character_agent(api_key: str, title: str, genre: str,
                    world: WorldInfo, plot_data: dict,
                    char_seed: str) -> List[Character]:
    log_section("CHARACTER AGENT — Building cast of characters")
    log("Generating protagonist, antagonist, and supporting cast …", "AGENT")

    system = """You are a character development specialist for fiction. You create
deeply layered, believable characters with clear motivations and compelling arcs.
Always respond with valid JSON only."""

    user = f"""Create the character roster for a {genre} novel titled "{title}".

World setting: {world.setting}
Plot logline: {plot_data.get('logline', '')}
Themes: {', '.join(plot_data.get('themes', []))}
Author's character notes: "{char_seed or 'None — use your creativity.'}"

Return a JSON object with a "characters" array. Include 4-6 characters:
{{
  "characters": [
    {{
      "name": "Full name",
      "role": "protagonist",
      "age": "Age or range",
      "personality": "Core traits, strengths, flaws (3-4 sentences)",
      "background": "Backstory relevant to the plot (2-3 sentences)",
      "goals": "What they want and why",
      "arc": "How they change from chapter 1 to final chapter"
    }}
  ]
}}

Make sure there is exactly 1 protagonist, 1 antagonist, and 2-4 supporting characters."""

    raw = call_llm(api_key, system, user,model=AGENT_MODELS["character"], max_tokens=5000)
    data = extract_json(raw)
    characters = [Character(**c) for c in data["characters"]]
    log(f"Created {len(characters)} characters: {', '.join(c.name for c in characters)}", "OK")
    db_log("character_agent_complete", str(len(characters)) + " characters")
    return characters


# ── 5D. OUTLINE AGENT ────────────────────────
def outline_agent(api_key: str, title: str, genre: str,
                  world: WorldInfo, characters: List[Character],
                  plot_data: dict, num_chapters: int) -> List[ChapterOutline]:
    log_section("OUTLINE AGENT — Creating chapter-by-chapter outline")
    log(f"Outlining {num_chapters} chapters …", "AGENT")

    char_names = [c.name for c in characters]
    plot_threads = [pt["thread_id"] + ": " + pt["description"]
                    for pt in plot_data.get("plot_threads", [])]

    system = """You are a novel outliner. You create tight, logical chapter outlines
that ensure great pacing and rising tension. Always respond with valid JSON only."""

    # We batch in groups of 5 to avoid token limits
    all_outlines = []
    batch_size = 5
    for batch_start in range(1, num_chapters + 1, batch_size):
        batch_end = min(batch_start + batch_size - 1, num_chapters)
        log(f"  Outlining chapters {batch_start}–{batch_end} …")

        user = f"""Create detailed chapter outlines for chapters {batch_start} through {batch_end}
of the {genre} novel "{title}".

Characters: {', '.join(char_names)}
Plot threads: {chr(10).join(plot_threads)}
Three-act structure: {json.dumps(plot_data.get('three_act_structure', {}))}
Major turning points: {json.dumps(plot_data.get('major_turning_points', []))}
World: {world.setting} | {world.time_period}
Total chapters in novel: {num_chapters}

Return a JSON object:
{{
  "chapters": [
    {{
      "chapter_number": {batch_start},
      "title": "Evocative chapter title",
      "summary": "What happens in this chapter (3-5 sentences)",
      "key_events": ["Event 1", "Event 2", "Event 3"],
      "characters_present": ["Name1", "Name2"],
      "plot_threads_advanced": ["PT1", "PT2"],
      "emotional_tone": "tense / hopeful / melancholic / etc."
    }}
    ... (all chapters {batch_start} to {batch_end})
  ]
}}"""

        raw = call_llm(api_key, system, user,model=AGENT_MODELS["outline"], max_tokens=5000)
        data = extract_json(raw)
        for ch in data["chapters"]:
            all_outlines.append(ChapterOutline(**ch))

    log(f"Outline complete — {len(all_outlines)} chapters outlined.", "OK")
    db_log("outline_agent_complete", f"{len(all_outlines)} chapters")
    return all_outlines


# ── 5E. CHAPTER AGENT ────────────────────────
def chapter_agent(api_key: str, bible: StoryBible,
                  outline: ChapterOutline,
                  previous_summary: str,
                  active_threads: List[str]) -> ChapterContent:
    log(f"Writing Chapter {outline.chapter_number}: '{outline.title}' …", "AGENT")

    chars_in_chapter = [c for c in bible.characters
                        if c.name in outline.characters_present]
    char_details = "\n".join(
        f"- {c.name} ({c.role}): {c.personality[:120]}" for c in chars_in_chapter)

    system = f"""You are an award-winning {bible.genre} novelist. You write immersive,
vivid prose with strong character voice, sensory detail, and emotional depth.
Write between 3500 and 5000 words.

Do not summarize.

Write full scenes.

Use dialogue.

Use internal thoughts.

Use descriptions.

Show events instead of telling them. Write genuine narrative prose —
not outlines, not summaries. Maintain consistent tone: {bible.world.atmosphere}."""

    user = f"""Write Chapter {outline.chapter_number} of "{bible.title}".

═══ STORY BIBLE ═══
Logline: {bible.logline}
World: {bible.world.setting} | {bible.world.time_period}
Geography: {bible.world.geography}
Society: {bible.world.society}
World rules: {bible.world.rules}
Themes: {', '.join(bible.themes)}

═══ CHARACTERS IN THIS CHAPTER ═══
{char_details}

═══ CHAPTER OUTLINE ═══
Title: {outline.title}
Summary: {outline.summary}
Key events: {', '.join(outline.key_events)}
Emotional tone: {outline.emotional_tone}
Plot threads to advance: {', '.join(outline.plot_threads_advanced)}

═══ PREVIOUS CHAPTER CONTEXT ═══
{previous_summary or "This is the opening chapter."}

═══ ACTIVE PLOT THREADS ═══
{chr(10).join(active_threads) if active_threads else "None yet."}

Write the full chapter now. Use section breaks (***) if needed.
After the chapter, on a new line write:
CHAPTER_SUMMARY: [2-3 sentence summary for the next chapter's context]
THREADS_UPDATED: [comma-separated list of thread IDs you advanced]"""

    raw = call_llm(api_key, system, user,model=AGENT_MODELS["chapter"], max_tokens=6000)

    # Parse out summary and threads footer
    summary_match = re.search(r"CHAPTER_SUMMARY:\s*(.+?)(?:THREADS_UPDATED:|$)",
                               raw, re.DOTALL)
    threads_match = re.search(r"THREADS_UPDATED:\s*(.+?)$", raw, re.DOTALL)

    chapter_summary = summary_match.group(1).strip() if summary_match else ""
    threads_updated = [t.strip() for t in threads_match.group(1).split(",")
                       ] if threads_match else []

    # Strip the footer from content
    content = re.sub(r"\nCHAPTER_SUMMARY:.*$", "", raw, flags=re.DOTALL).strip()
    word_count = len(content.split())

    log(f"  Chapter {outline.chapter_number} drafted — {word_count} words.", "OK")
    return ChapterContent(
        chapter_number=outline.chapter_number,
        title=outline.title,
        content=content,
        word_count=word_count,
        summary=chapter_summary,
        plot_threads_updated=threads_updated
    )


# ── 5F. EDITOR AGENT ─────────────────────────
def editor_agent(api_key: str, chapter: ChapterContent,
                 bible: StoryBible) -> EditedChapter:
    log(f"  Editing Chapter {chapter.chapter_number} …", "AGENT")

    system = """You are a professional fiction editor. You polish prose, fix
inconsistencies, improve flow, and strengthen voice — without changing the story.
Return the improved chapter text followed by brief editor notes."""

    user = f"""Edit Chapter {chapter.chapter_number}: "{chapter.title}" of "{bible.title}".

Genre: {bible.genre} | Tone: {bible.world.atmosphere}

DRAFT:
{chapter.content}

Instructions:
- Fix grammar, punctuation, awkward phrasing
- Strengthen sensory details and dialogue
- Ensure consistent character voice
- Do NOT add new plot points or change events
- Keep the same approximate length or longer

After the edited chapter write:
EDITOR_NOTES: [2-3 sentences on what you improved]"""

    raw = call_llm(api_key, system, user,model=AGENT_MODELS["editor"], max_tokens=6000)

    notes_match = re.search(r"EDITOR_NOTES:\s*(.+?)$", raw, re.DOTALL)
    editor_notes = notes_match.group(1).strip() if notes_match else "Light polish applied."
    edited_content = re.sub(r"\nEDITOR_NOTES:.*$", "", raw, flags=re.DOTALL).strip()
    word_count = len(edited_content.split())

    log(f"  Chapter {chapter.chapter_number} edited — {word_count} words.", "OK")
    return EditedChapter(
        chapter_number=chapter.chapter_number,
        title=chapter.title,
        content=edited_content,
        word_count=word_count,
        editor_notes=editor_notes
    )


# ── 5G. REVIEW AGENT ─────────────────────────
def review_agent(api_key: str, bible: StoryBible,
                 chapters: List[dict]) -> ReviewReport:
    log_section("REVIEW AGENT — Final manuscript review")
    log("Analysing full manuscript for consistency and quality …", "AGENT")

    total_words = sum(c["word_count"] for c in chapters)
    chapter_titles = "\n".join(f"Ch {c['chapter_number']}: {c['title']} ({c['word_count']} words)"
                                for c in chapters)

    system = """You are a senior literary editor reviewing a completed novel manuscript.
Provide honest, constructive feedback on the whole work."""

    user = f"""Review the completed novel "{bible.title}" ({bible.genre}).

Total word count: {total_words:,}
Logline: {bible.logline}
Themes: {', '.join(bible.themes)}

Chapter list:
{chapter_titles}

Character summaries:
{chr(10).join(f"- {c.name} ({c.role}): arc = {c.arc}" for c in bible.characters)}

Based on the structure and metadata above, provide a review as JSON:
{{
  "overall_score": 8,
  "consistency_notes": "Assessment of plot and character consistency",
  "pacing_notes": "Assessment of story pacing across chapters",
  "character_arc_notes": "How well characters developed",
  "final_recommendation": "Overall verdict and what makes this novel work"
}}"""

    raw = call_llm(api_key, system, user,model=AGENT_MODELS["reviewer"], max_tokens=1000)
    data = extract_json(raw)
    report = ReviewReport(**data)
    log(f"Review complete. Overall score: {report.overall_score}/10", "OK")
    db_log("review_agent_complete", f"score={report.overall_score}")
    return report


# ─────────────────────────────────────────────
# 6. PDF EXPORT
# ─────────────────────────────────────────────
def export_pdf(bible: StoryBible, chapters: List[dict],
               review: ReviewReport, output_path: str):
    log_section("PDF EXPORT — Assembling final novel PDF")
    log(f"Output path: {output_path}", "PDF")

    doc = SimpleDocTemplate(
        output_path,
        pagesize=letter,
        rightMargin=1.2 * inch,
        leftMargin=1.2 * inch,
        topMargin=1.0 * inch,
        bottomMargin=1.0 * inch,
        title=bible.title,
        author="Multi-Agent Novel Generator"
    )

    styles = getSampleStyleSheet()

    # Custom styles
    title_style = ParagraphStyle(
        "NovelTitle",
        parent=styles["Title"],
        fontSize=28,
        spaceAfter=12,
        textColor=colors.HexColor("#1a1a2e"),
        alignment=TA_CENTER,
        fontName="Times-Bold"
    )
    subtitle_style = ParagraphStyle(
        "NovelSubtitle",
        parent=styles["Normal"],
        fontSize=14,
        spaceAfter=6,
        textColor=colors.HexColor("#4a4a8a"),
        alignment=TA_CENTER,
        fontName="Times-Italic"
    )
    chapter_heading_style = ParagraphStyle(
        "ChapterHeading",
        parent=styles["Heading1"],
        fontSize=18,
        spaceBefore=18,
        spaceAfter=10,
        textColor=colors.HexColor("#1a1a2e"),
        alignment=TA_CENTER,
        fontName="Times-Bold"
    )
    chapter_num_style = ParagraphStyle(
        "ChapterNum",
        parent=styles["Normal"],
        fontSize=11,
        spaceAfter=4,
        textColor=colors.HexColor("#888888"),
        alignment=TA_CENTER,
        fontName="Times-Italic"
    )
    body_style = ParagraphStyle(
        "NovelBody",
        parent=styles["Normal"],
        fontSize=11,
        leading=17,
        spaceAfter=8,
        alignment=TA_JUSTIFY,
        fontName="Times-Roman",
        firstLineIndent=24
    )
    section_break_style = ParagraphStyle(
        "SectionBreak",
        parent=styles["Normal"],
        fontSize=12,
        spaceBefore=12,
        spaceAfter=12,
        alignment=TA_CENTER,
        fontName="Times-Roman"
    )
    bible_heading_style = ParagraphStyle(
        "BibleHeading",
        parent=styles["Heading2"],
        fontSize=13,
        spaceBefore=12,
        spaceAfter=6,
        textColor=colors.HexColor("#2c2c6c"),
        fontName="Times-Bold"
    )
    bible_body_style = ParagraphStyle(
        "BibleBody",
        parent=styles["Normal"],
        fontSize=10,
        leading=14,
        spaceAfter=6,
        fontName="Times-Roman"
    )
    notes_style = ParagraphStyle(
        "Notes",
        parent=styles["Normal"],
        fontSize=9,
        leading=13,
        spaceAfter=4,
        textColor=colors.HexColor("#555555"),
        fontName="Times-Italic"
    )

    story = []

    def safe_para(text: str, style) -> Paragraph:
        """Escape XML special chars for ReportLab."""
        text = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
        # restore *** section breaks
        text = text.replace("***", "* * *")
        return Paragraph(text, style)

    # ── TITLE PAGE ──────────────────────────
    story.append(Spacer(1, 1.5 * inch))
    story.append(safe_para(bible.title, title_style))
    story.append(Spacer(1, 0.15 * inch))
    story.append(safe_para(f"A {bible.genre} Novel", subtitle_style))
    story.append(Spacer(1, 0.3 * inch))
    story.append(HRFlowable(width="60%", thickness=1,
                             color=colors.HexColor("#4a4a8a"),
                             hAlign="CENTER"))
    story.append(Spacer(1, 0.3 * inch))
    story.append(safe_para(f'"{bible.logline}"', subtitle_style))
    story.append(Spacer(1, 1.0 * inch))
    total_words = sum(c["word_count"] for c in chapters)
    story.append(safe_para(
        f"Generated by Multi-Agent Novel Generator<br/>"
        f"{len(chapters)} Chapters  |  {total_words:,} Words<br/>"
        f"{datetime.now().strftime('%B %d, %Y')}",
        subtitle_style))
    story.append(PageBreak())

    # ── STORY BIBLE ──────────────────────────
    story.append(safe_para("Story Bible", chapter_heading_style))
    story.append(HRFlowable(width="80%", thickness=0.5,
                             color=colors.HexColor("#cccccc"), hAlign="CENTER"))
    story.append(Spacer(1, 0.2 * inch))

    story.append(safe_para("Themes", bible_heading_style))
    story.append(safe_para(", ".join(bible.themes), bible_body_style))

    story.append(safe_para("World", bible_heading_style))
    story.append(safe_para(f"<b>Setting:</b> {bible.world.setting}", bible_body_style))
    story.append(safe_para(f"<b>Time Period:</b> {bible.world.time_period}", bible_body_style))
    story.append(safe_para(f"<b>Geography:</b> {bible.world.geography}", bible_body_style))
    story.append(safe_para(f"<b>Society:</b> {bible.world.society}", bible_body_style))
    story.append(safe_para(f"<b>Rules:</b> {bible.world.rules}", bible_body_style))
    story.append(safe_para(f"<b>Atmosphere:</b> {bible.world.atmosphere}", bible_body_style))

    story.append(safe_para("Characters", bible_heading_style))
    for char in bible.characters:
        story.append(safe_para(
            f"<b>{char.name}</b> ({char.role.upper()}{', age ' + char.age if char.age else ''})",
            bible_body_style))
        story.append(safe_para(f"<i>Personality:</i> {char.personality}", notes_style))
        story.append(safe_para(f"<i>Background:</i> {char.background}", notes_style))
        story.append(safe_para(f"<i>Goals:</i> {char.goals}", notes_style))
        story.append(safe_para(f"<i>Arc:</i> {char.arc}", notes_style))
        story.append(Spacer(1, 0.05 * inch))

    story.append(safe_para("Plot Threads", bible_heading_style))
    for pt in bible.plot_threads:
        story.append(safe_para(
            f"<b>[{pt.thread_id}]</b> {pt.description} — Status: {pt.status}",
            bible_body_style))

    story.append(PageBreak())

    # ── TABLE OF CONTENTS ────────────────────
    story.append(safe_para("Table of Contents", chapter_heading_style))
    story.append(HRFlowable(width="80%", thickness=0.5,
                             color=colors.HexColor("#cccccc"), hAlign="CENTER"))
    story.append(Spacer(1, 0.2 * inch))
    for ch in chapters:
        story.append(safe_para(
            f"Chapter {ch['chapter_number']}: {ch['title']}"
            f"  ({ch['word_count']:,} words)",
            bible_body_style))
    story.append(Spacer(1, 0.2 * inch))
    story.append(safe_para(f"<b>Total: {total_words:,} words across {len(chapters)} chapters</b>",
                           bible_body_style))
    story.append(PageBreak())

    # ── CHAPTERS ─────────────────────────────
    for ch in chapters:
        log(f"  Adding Chapter {ch['chapter_number']} to PDF …", "PDF")

        story.append(safe_para(f"Chapter {ch['chapter_number']}", chapter_num_style))
        story.append(safe_para(ch["title"], chapter_heading_style))
        story.append(HRFlowable(width="50%", thickness=0.5,
                                 color=colors.HexColor("#cccccc"), hAlign="CENTER"))
        story.append(Spacer(1, 0.25 * inch))

        content = ch["content"]
        # Split on section breaks
        sections = re.split(r"\n\s*\*\s*\*\s*\*\s*\n", content)
        for i, section in enumerate(sections):
            if i > 0:
                story.append(safe_para("* * *", section_break_style))
            paragraphs = [p.strip() for p in section.split("\n\n") if p.strip()]
            for para_text in paragraphs:
                # Single newlines within a paragraph become spaces
                para_text = para_text.replace("\n", " ")
                story.append(safe_para(para_text, body_style))

        # Editor notes as small annotation
        if ch.get("editor_notes"):
            story.append(Spacer(1, 0.1 * inch))
            story.append(safe_para(f"[Editor: {ch['editor_notes']}]", notes_style))

        story.append(PageBreak())

    # ── REVIEW REPORT ────────────────────────
    story.append(safe_para("Editorial Review", chapter_heading_style))
    story.append(HRFlowable(width="80%", thickness=0.5,
                             color=colors.HexColor("#cccccc"), hAlign="CENTER"))
    story.append(Spacer(1, 0.2 * inch))
    story.append(safe_para(f"Overall Score: {review.overall_score}/10", bible_heading_style))
    story.append(safe_para(f"<b>Consistency:</b> {review.consistency_notes}", bible_body_style))
    story.append(safe_para(f"<b>Pacing:</b> {review.pacing_notes}", bible_body_style))
    story.append(safe_para(f"<b>Character Arcs:</b> {review.character_arc_notes}", bible_body_style))
    story.append(safe_para(f"<b>Final Verdict:</b> {review.final_recommendation}", bible_body_style))

    doc.build(story)
    size_kb = os.path.getsize(output_path) // 1024
    log(f"PDF written: {output_path} ({size_kb} KB, {total_words:,} words)", "OK")


# ─────────────────────────────────────────────
# 7. MAIN ORCHESTRATOR
# ─────────────────────────────────────────────
def main():

    # ── USER INPUTS ──────────────────────────
    print("─" * LOG_WIDTH)
    print("  CONFIGURATION")
    print("─" * LOG_WIDTH)

    api_key = input("OpenRouter API Key: ").strip()
    if not api_key:
        sys.exit("❌  API key is required.")

    title = input("Novel Title: ").strip() or "The Last Signal"
    genre = input("Genre (e.g. fantasy, sci-fi, thriller): ").strip() or "fantasy"
    plot_seed = input("Plot seed (optional, press Enter to skip): ").strip()
    char_seed = input("Character notes (optional, press Enter to skip): ").strip()

    try:
        num_chapters = int(input("Number of chapters [25]: ").strip() or "25")
    except ValueError:
        num_chapters = 25

    safe_title = re.sub(r"[^\w\s-]", "", title).strip().replace(" ", "_")
    pdf_path = f"{safe_title}_novel.pdf"

    log_banner(f'"{title}" — {genre.title()} — {num_chapters} chapters')

    # ── INIT ─────────────────────────────────
    init_db()
    db_log("run_start", f"{title} | {genre} | {num_chapters} chapters")

    # ── CHECK FOR EXISTING BIBLE ──────────────
    bible = load_story_bible()
    if bible:
        log("Resuming from saved Story Bible.", "OK")
    else:
        log_section("PHASE 1 — Building Story Foundation")

        # Plot Agent
        plot_data = plot_agent(api_key, title, genre, plot_seed, num_chapters)
        time.sleep(1)

        # World Agent
        world = world_agent(api_key, title, genre, plot_data)
        time.sleep(1)

        # Character Agent
        characters = character_agent(api_key, title, genre, world, plot_data, char_seed)
        time.sleep(1)

        # Outline Agent
        outlines = outline_agent(api_key, title, genre, world, characters,
                                  plot_data, num_chapters)
        time.sleep(1)

        # Assemble Story Bible
        plot_threads = [PlotThread(**pt) for pt in plot_data.get("plot_threads", [])]
        bible = StoryBible(
            title=title,
            genre=genre,
            logline=plot_data.get("logline", ""),
            themes=plot_data.get("themes", []),
            world=world,
            characters=characters,
            plot_threads=plot_threads,
            chapter_outlines=outlines
        )
        save_story_bible(bible)
        log("Story Bible complete and saved.", "OK")

    # ── PHASE 2: GENERATE CHAPTERS ────────────
    log_section("PHASE 2 — Writing & Editing Chapters")
    total_chapters = len(bible.chapter_outlines)
    log(f"Will generate {total_chapters} chapters.")

    previous_summary = ""
    active_threads = [f"{pt.thread_id}: {pt.description}"
                      for pt in bible.plot_threads if pt.status == "active"]

    for outline in bible.chapter_outlines:
        n = outline.chapter_number
        log(f"\n{'─'*60}")
        log(f"CHAPTER {n}/{total_chapters}: '{outline.title}'", "CHAPTER")

        if chapter_exists(n):
            log(f"  Chapter {n} already done — loading summary from DB.", "OK")
            done = load_all_chapters()
            ch_done = next((c for c in done if c["chapter_number"] == n), None)
            if ch_done:
                previous_summary = ch_done.get("summary", "")
                for t in ch_done.get("plot_threads", []):
                    if t not in active_threads:
                        active_threads.append(t)
            continue

        log(f"  Active plot threads: {', '.join(active_threads[:3])} …")
        log(f"  Previous summary: {previous_summary[:80]}…" if previous_summary else
            "  This is the first chapter.")

        # Chapter Agent
        chapter = chapter_agent(api_key, bible, outline,
                                 previous_summary, active_threads)
        time.sleep(1)

        # Editor Agent
        edited = editor_agent(api_key, chapter, bible)
        time.sleep(0.5)

        # Update state
        for t in chapter.plot_threads_updated:
            entry = f"{t}: advanced in chapter {n}"
            if entry not in active_threads:
                active_threads.append(entry)

        # Save to DB
        save_chapter(edited, chapter.summary, chapter.plot_threads_updated)
        previous_summary = chapter.summary

        words_so_far = sum(c["word_count"] for c in load_all_chapters())
        log(f"  ✅  Chapter {n} done. Cumulative total: {words_so_far:,} words.", "CHAPTER")
        db_log(f"chapter_{n}_done", f"{edited.word_count} words")

        # Small pause between chapters to be kind to the API
        time.sleep(2)

    # ── PHASE 3: REVIEW ──────────────────────
    log_section("PHASE 3 — Final Review")
    all_chapters = load_all_chapters()
    review = review_agent(api_key, bible, all_chapters)

    # ── PHASE 4: PDF ─────────────────────────
    export_pdf(bible, all_chapters, review, pdf_path)

    # ── FINAL STATS ──────────────────────────
    total_words = sum(c["word_count"] for c in all_chapters)
    log_banner("NOVEL GENERATION COMPLETE")
    log(f"Title          : {bible.title}", "OK")
    log(f"Genre          : {bible.genre}", "OK")
    log(f"Chapters       : {len(all_chapters)}", "OK")
    log(f"Total words    : {total_words:,}", "OK")
    log(f"Review score   : {review.overall_score}/10", "OK")
    log(f"PDF output     : {pdf_path}", "PDF")
    log(f"Database       : {DB_PATH}", "DB")
    db_log("run_complete", f"{total_words} words | {len(all_chapters)} chapters")
    print()


if __name__ == "__main__":
    main()

────────────────────────────────────────────────────────────────────────
  CONFIGURATION
────────────────────────────────────────────────────────────────────────
OpenRouter API Key: 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
Novel Title: Funny Bunny
Genre (e.g. fantasy, sci-fi, thriller): Funny
Plot seed (optional, press Enter to skip): 
Character notes (optional, press Enter to skip): 
Number of chapters [25]: 12

████████████████████████████████████████████████████████████████████████
                  "Funny Bunny" — Funny — 12 chapters                   
████████████████████████████████████████████████████████████████████████

[20:21:35] 💾  Initialising SQLite database …
[20:21:35] 💾  Database ready.

════════════════════════════════════════════════════════════════════════
  PHASE 1 — Building Story Foundation
════════════════════════════════════════════════════════════════════════

═════════════════════════════════════════════════════════════════════